# Telegram Bot 库详细教程

本教程介绍如何使用 `python-telegram-bot` 库创建和管理 Telegram 机器人。

## 1. 安装依赖

In [ ]:
# 安装必要的库
!pip install python-telegram-bot requests pysocks

## 2. 基础配置

In [ ]:
import requests
from telegram import Update, Bot
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
import asyncio
import json

# 配置信息
BOT_TOKEN = "YOUR_BOT_TOKEN"  # 替换为你的机器人token
CHAT_ID = "YOUR_CHAT_ID"     # 替换为你的聊天ID
PROXY_URL = "socks5://192.168.0.103:7897"  # 代理配置

## 3. 简单消息发送

In [ ]:
def send_simple_message(token: str, chat_id: str, message: str) -> bool:
    """发送简单消息"""
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    
    payload = {
        'chat_id': chat_id,
        'text': message,
        'parse_mode': 'Markdown'
    }
    
    proxies = {
        'http': PROXY_URL,
        'https': PROXY_URL
    } if PROXY_URL else None
    
    try:
        response = requests.post(url, json=payload, proxies=proxies, timeout=30)
        response.raise_for_status()
        return response.json().get('ok', False)
    except Exception as e:
        print(f"发送失败: {e}")
        return False

# 测试发送消息
if BOT_TOKEN != "YOUR_BOT_TOKEN":
    success = send_simple_message(BOT_TOKEN, CHAT_ID, "🤖 测试消息")
    print(f"发送结果: {success}")
else:
    print("请先配置BOT_TOKEN和CHAT_ID")

## 4. 格式化消息发送

In [ ]:
def send_formatted_message(token: str, chat_id: str) -> bool:
    """发送格式化消息"""
    message = """
📊 **加密货币监控报告**

🔥 **交易量前5名**
```
排名 币种            价格         24h交易量       24h涨跌
------------------------------------------------------------
1    BTC/USDT       $43250.0000  $2,450,000,000  2.45%
2    ETH/USDT       $2650.0000   $1,200,000,000  1.85%
3    SOL/USDT       $98.5000     $800,000,000    3.20%
4    BNB/USDT       $315.0000    $450,000,000    0.95%
5    XRP/USDT       $0.6200      $400,000,000    -1.25%
```

⚡ **波动前5名**
```
排名 币种            价格         24h涨跌    高点         低点
----------------------------------------------------------------------
1    DOGE/USDT      $0.0850      15.25%    $0.0920      $0.0780
2    SHIB/USDT      $0.0000      12.80%    $0.0000      $0.0000
3    PEPE/USDT      $0.0000      -10.50%   $0.0000      $0.0000
4    WIF/USDT       $2.1500      8.90%     $2.3500      $1.9800
5    BONK/USDT      $0.0000      -7.65%    $0.0000      $0.0000
```

🕐 **更新时间**: 2024-01-15 14:30:00
"""
    
    return send_simple_message(token, chat_id, message)

# 测试发送格式化消息
if BOT_TOKEN != "YOUR_BOT_TOKEN":
    success = send_formatted_message(BOT_TOKEN, CHAT_ID)
    print(f"格式化消息发送结果: {success}")

## 5. 长消息分割发送

In [ ]:
def send_long_message(token: str, chat_id: str, long_message: str) -> bool:
    """发送长消息（自动分割）"""
    max_length = 4000  # Telegram限制4096字符
    
    if len(long_message) <= max_length:
        return send_simple_message(token, chat_id, long_message)
    
    # 分割消息
    parts = []
    current_part = ""
    
    for line in long_message.split('\n'):
        if len(current_part + line + '\n') > max_length:
            if current_part:
                parts.append(current_part)
                current_part = line + '\n'
            else:
                parts.append(line[:max_length])
                current_part = line[max_length:] + '\n'
        else:
            current_part += line + '\n'
    
    if current_part:
        parts.append(current_part)
    
    # 发送所有部分
    success = True
    for i, part in enumerate(parts):
        part_message = f"📊 报告 {i+1}/{len(parts)}\n\n{part}"
        part_success = send_simple_message(token, chat_id, part_message)
        success = success and part_success
        
        if i < len(parts) - 1:  # 避免发送过快
            import time
            time.sleep(1)
    
    return success

# 测试长消息
long_text = "\n".join([f"第{i}行数据: {'测试内容' * 20}" for i in range(100)])
print(f"长消息长度: {len(long_text)} 字符")

if BOT_TOKEN != "YOUR_BOT_TOKEN":
    success = send_long_message(BOT_TOKEN, CHAT_ID, long_text)
    print(f"长消息发送结果: {success}")

## 6. 创建交互式机器人

In [ ]:
class TelegramBot:
    """交互式Telegram机器人"""
    
    def __init__(self, token: str, proxy_url: str = None):
        self.token = token
        self.proxy_url = proxy_url
        
        # 配置应用
        builder = Application.builder().token(token)
        if proxy_url:
            builder.proxy(proxy_url).get_updates_proxy(proxy_url)
            builder.connect_timeout(30).read_timeout(30).write_timeout(30)
        
        self.app = builder.build()
        
        # 注册处理器
        self.app.add_handler(CommandHandler("start", self.start_command))
        self.app.add_handler(CommandHandler("help", self.help_command))
        self.app.add_handler(CommandHandler("monitor", self.monitor_command))
        self.app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, self.handle_message))
    
    async def start_command(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """处理/start命令"""
        welcome_msg = """
🤖 **欢迎使用加密货币监控机器人!**

📋 **可用命令:**
/start - 显示欢迎信息
/help - 显示帮助信息
/monitor - 获取市场监控报告

💬 **直接发送消息** - 我会回复你
"""
        await update.message.reply_text(welcome_msg, parse_mode='Markdown')
    
    async def help_command(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """处理/help命令"""
        help_msg = """
🆘 **帮助信息**

这是一个加密货币监控机器人，可以:
• 📊 提供实时市场数据
• 📈 分析交易量和波动性
• 🔔 发送定时监控报告

使用 /monitor 获取最新市场报告
"""
        await update.message.reply_text(help_msg, parse_mode='Markdown')
    
    async def monitor_command(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """处理/monitor命令"""
        # 发送"正在获取数据"提示
        thinking_msg = await update.message.reply_text("📊 正在获取市场数据...")
        
        # 模拟获取数据
        await asyncio.sleep(2)
        
        # 删除提示消息并发送报告
        await thinking_msg.delete()
        
        report = """
📊 **实时市场监控报告**

🔥 **热门币种**
• BTC/USDT: $43,250 (+2.45%)
• ETH/USDT: $2,650 (+1.85%)
• SOL/USDT: $98.50 (+3.20%)

⚡ **高波动币种**
• DOGE/USDT: +15.25%
• SHIB/USDT: +12.80%
• PEPE/USDT: -10.50%

🕐 更新时间: 刚刚
"""
        
        await update.message.reply_text(report, parse_mode='Markdown')
    
    async def handle_message(self, update: Update, context: ContextTypes.DEFAULT_TYPE):
        """处理普通消息"""
        user_message = update.message.text
        
        # 简单的回复逻辑
        if "价格" in user_message or "行情" in user_message:
            reply = "📈 当前BTC价格: $43,250 (+2.45%)\n使用 /monitor 获取完整报告"
        elif "你好" in user_message or "hello" in user_message.lower():
            reply = "👋 你好！我是加密货币监控机器人，使用 /help 查看帮助"
        else:
            reply = f"🤖 你说: {user_message}\n\n使用 /help 查看可用命令"
        
        await update.message.reply_text(reply)
    
    def run(self):
        """启动机器人"""
        print("🚀 机器人启动中...")
        self.app.run_polling(allowed_updates=Update.ALL_TYPES)

# 创建机器人实例（不自动运行）
if BOT_TOKEN != "YOUR_BOT_TOKEN":
    bot = TelegramBot(BOT_TOKEN, PROXY_URL)
    print("✅ 机器人已创建，使用 bot.run() 启动")
    # bot.run()  # 取消注释以启动机器人
else:
    print("请先配置BOT_TOKEN")

## 7. 获取聊天ID

In [ ]:
def get_chat_id(token: str) -> list:
    """获取机器人的聊天ID列表"""
    url = f"https://api.telegram.org/bot{token}/getUpdates"
    
    proxies = {
        'http': PROXY_URL,
        'https': PROXY_URL
    } if PROXY_URL else None
    
    try:
        response = requests.get(url, proxies=proxies, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        chat_ids = []
        for update in data.get('result', []):
            if 'message' in update:
                chat_id = update['message']['chat']['id']
                chat_name = update['message']['chat'].get('first_name', 'Unknown')
                chat_ids.append({'id': chat_id, 'name': chat_name})
        
        return chat_ids
    except Exception as e:
        print(f"获取聊天ID失败: {e}")
        return []

# 获取聊天ID
if BOT_TOKEN != "YOUR_BOT_TOKEN":
    chat_ids = get_chat_id(BOT_TOKEN)
    print("📱 聊天ID列表:")
    for chat in chat_ids:
        print(f"  ID: {chat['id']}, 名称: {chat['name']}")
else:
    print("请先配置BOT_TOKEN")

## 8. 机器人信息查询

In [ ]:
def get_bot_info(token: str) -> dict:
    """获取机器人信息"""
    url = f"https://api.telegram.org/bot{token}/getMe"
    
    proxies = {
        'http': PROXY_URL,
        'https': PROXY_URL
    } if PROXY_URL else None
    
    try:
        response = requests.get(url, proxies=proxies, timeout=30)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print(f"获取机器人信息失败: {e}")
        return {}

# 查询机器人信息
if BOT_TOKEN != "YOUR_BOT_TOKEN":
    bot_info = get_bot_info(BOT_TOKEN)
    if bot_info.get('ok'):
        result = bot_info['result']
        print("🤖 机器人信息:")
        print(f"  ID: {result['id']}")
        print(f"  用户名: @{result['username']}")
        print(f"  名称: {result['first_name']}")
        print(f"  是否机器人: {result['is_bot']}")
    else:
        print("❌ 获取机器人信息失败")
else:
    print("请先配置BOT_TOKEN")

## 9. 完整的通知类

In [ ]:
class TelegramNotifier:
    """完整的Telegram通知类"""
    
    def __init__(self, token: str, chat_id: str, proxy_url: str = None):
        self.token = token
        self.chat_id = chat_id
        self.base_url = f"https://api.telegram.org/bot{token}"
        self.proxies = {
            'http': proxy_url,
            'https': proxy_url
        } if proxy_url else None
    
    def send_message(self, message: str, parse_mode: str = "Markdown") -> bool:
        """发送消息"""
        url = f"{self.base_url}/sendMessage"
        payload = {
            'chat_id': self.chat_id,
            'text': message,
            'parse_mode': parse_mode
        }
        
        try:
            response = requests.post(url, json=payload, proxies=self.proxies, timeout=30)
            response.raise_for_status()
            return response.json().get('ok', False)
        except Exception as e:
            print(f"发送失败: {e}")
            return False
    
    def send_photo(self, photo_path: str, caption: str = "") -> bool:
        """发送图片"""
        url = f"{self.base_url}/sendPhoto"
        
        try:
            with open(photo_path, 'rb') as photo:
                files = {'photo': photo}
                data = {
                    'chat_id': self.chat_id,
                    'caption': caption
                }
                response = requests.post(url, files=files, data=data, proxies=self.proxies, timeout=30)
                response.raise_for_status()
                return response.json().get('ok', False)
        except Exception as e:
            print(f"发送图片失败: {e}")
            return False
    
    def send_document(self, document_path: str, caption: str = "") -> bool:
        """发送文档"""
        url = f"{self.base_url}/sendDocument"
        
        try:
            with open(document_path, 'rb') as document:
                files = {'document': document}
                data = {
                    'chat_id': self.chat_id,
                    'caption': caption
                }
                response = requests.post(url, files=files, data=data, proxies=self.proxies, timeout=30)
                response.raise_for_status()
                return response.json().get('ok', False)
        except Exception as e:
            print(f"发送文档失败: {e}")
            return False

# 创建通知器实例
if BOT_TOKEN != "YOUR_BOT_TOKEN" and CHAT_ID != "YOUR_CHAT_ID":
    notifier = TelegramNotifier(BOT_TOKEN, CHAT_ID, PROXY_URL)
    print("✅ 通知器已创建")
    
    # 测试发送
    success = notifier.send_message("📚 **Telegram教程测试消息**\n\n这是一条测试消息！")
    print(f"测试消息发送结果: {success}")
else:
    print("请先配置BOT_TOKEN和CHAT_ID")

## 10. 使用总结

### 基本步骤:
1. **创建机器人**: 在Telegram中找到@BotFather，发送/newbot创建机器人
2. **获取Token**: BotFather会给你一个token
3. **获取Chat ID**: 向机器人发送消息，然后调用getUpdates API获取chat_id
4. **配置代理**: 如果需要，配置socks5代理
5. **发送消息**: 使用requests或python-telegram-bot库发送消息

### 常用功能:
- ✅ 发送文本消息（支持Markdown格式）
- ✅ 发送图片和文档
- ✅ 长消息自动分割
- ✅ 交互式机器人（命令处理）
- ✅ 代理支持
- ✅ 错误处理

### 注意事项:
- 消息长度限制: 4096字符
- 发送频率限制: 每秒最多30条消息
- 代理配置: 国内需要配置代理访问Telegram API
- Token安全: 不要泄露机器人token